# 计算图框架

QuantStudio 系统以计算图为核心，主要计算以有向无环图（DAG）的形式表达。每个节点完成一个计算单元，边表示依赖关系：若节点 A 依赖节点 B，引擎沿 A → B 传递请求，再沿 B → A 收集结果。计算引擎负责调度和执行计算图。

![QuantStudio系统](https://gitee.com/scorpi000/QSImage/raw/master/QuantStudio/QuantStudio系统.jpg)

计算图将一个任务拆成若干节点：节点声明参数和依赖，引擎计算依赖后将结果交给当前节点。例如计算 `1 + 2`，可以先创建两个数字节点，再连接到一个求和节点。

本篇演示创建、连接、运行节点，以及复用节点配置。只使用现成因子或回测模块时，通常由对应模块构建计算图；需要扩展计算单元时再编写 `Node` 子类。

[计算引擎](计算引擎.ipynb)介绍选型与失败处理，[缓存](缓存.ipynb)介绍数据读写和跨运行复用。[原理和细节](原理和细节.ipynb)集中说明调度、身份、上下文和依赖模型的实现约定。

## 创建并运行一个计算图

自定义节点继承 `Node`，一般只需实现 `backward_compute()`。影响计算行为的配置放在 `__QS_ArgClass__` 中；依赖放在 `__QS_DepClass__` 中。这样才能支持参数校验、模板构造和正确的计算身份。

下面的 `Number` 声明数值参数，`Add` 声明首项和其他加数。`inputs.first`、`inputs.others` 接收的是依赖的计算结果。

In [1]:
import logging
from QuantStudio.Core import setDefaultLogLevel

setDefaultLogLevel(logging.WARNING)

In [2]:
from pydantic import Field
from QuantStudio.Core.api import Node, DependencyModel, Context, Engine


class Number(Node):
    """返回参数中声明的数值。"""

    class __QS_DepClass__(DependencyModel):
        """数字节点没有依赖。"""

    class __QS_ArgClass__(Node.__QS_ArgClass__):
        Value: float = Field(frozen=True, title="数值")

    def backward_compute(self, path, inputs, context, local_context=None):
        """返回数值。

        Args:
            path: 当前运行路径。
            inputs: 无依赖的结果视图。
            context: 本次运行上下文。
            local_context: 局部上下文。

        Returns:
            参数声明的数值。
        """
        return self.Args.Value


class Add(Node):
    """将首项与其余各项相加。"""

    class __QS_DepClass__(DependencyModel):
        first: Node = Field(title="首项")
        others: tuple[Node, ...] = Field(min_length=1, title="其他项")

    def backward_compute(self, path, inputs, context, local_context=None):
        """汇总依赖结果。

        Args:
            path: 当前运行路径。
            inputs: 按 first、others 组织的依赖结果。
            context: 本次运行上下文。
            local_context: 局部上下文。

        Returns:
            所有依赖结果的和。
        """
        return inputs.first + sum(inputs.others)

In [3]:
one = Number(args={"Value": 1, "Name": "一"})
two = Number(args={"Value": 2, "Name": "二"})
addition = Add(deps={"first": one, "others": (two,)}, args={"Name": "一加二"})

engine = Engine()
context = Context()
results = engine.run([addition], context)
assert results == [3]
print(results)  # [3.0]

[3.0]


`run()` 接收目标节点列表，返回与目标顺序一致的结果列表。只需传入最终目标，其依赖由引擎自动处理；也可以一次传入多个目标。

`Name` 用于显示。`Value` 这样的业务参数通过 `args` 指定，构造时按 Pydantic 字段定义校验。参数的一般约定见[通则和约定](../通则和约定.ipynb)。

## 连接依赖与复用配置

声明了命名依赖的节点，使用 `deps={字段名: 节点或节点元组}` 连接。上例中 `first` 是单个节点，`others` 至少包含一个节点。

- `node.Dependencies` 可读取完整连接关系，例如 `addition.Dependencies.first`。
- `node.Deps` 是展开后的依赖列表；修改它不会改变节点连接。需要替换依赖时，用 `new(deps=...)` 创建新节点。
- 未自定义依赖模型的普通 `Node` 使用默认 `nodes` 槽位，也允许通过列表或元组传入依赖。
- `new(args=...)` 复用已有参数和依赖，创建一个新实例；它不会修改原节点。

In [4]:
three = one.new(args={"Value": 3, "Name": "三"})
second_addition = addition.new(
    deps={"first": three, "others": (two,)},
    args={"Name": "三加二"},
)
assert addition.Dependencies.first is one
results = engine.run([addition, second_addition], Context())
assert results == [3, 5]
print(results)  # [3.0, 5.0]

[3.0, 5.0]


## 分步配置：节点模板

如果依赖或参数尚未齐全，先使用 `template()`，补齐后再 `build()`。模板不能直接交给引擎。

`bind()` 替换指定依赖，`with_args()` 更新参数，`unbind()` 移除绑定；这些操作都返回新模板。`missing_dependencies` 列出未绑定的必填槽位，`validate()` 返回完整校验诊断，空元组表示校验通过。

In [5]:
template = Add.template(args={"Name": "待配置的加法"}, deps={"first": one})
print(template.missing_dependencies)  # ('others',)
completed = template.bind(others=(two, three))
assert template.missing_dependencies == ("others",)
assert completed.validate() == ()

configured = completed.build()
results = engine.run([configured], Context())
assert results == [6]
print(results)  # [6.0]

('others',)
[6.0]


## 运行请求与上下文

`Context` 保存本次运行共享的信息。日期、资产等单次计算请求由 `fwd_data_list` 传入，与目标节点一一对应。需要单独指定初始化范围时，再传 `init_data_list`；省略它时使用 `fwd_data_list`。

```python
results = engine.run(
    node_list,
    context,
    fwd_data_list=requests,
    init_data_list=initialization,
)
```

上面的调用形式用于已有业务节点；本篇的数字节点不需要请求，直接省略两个列表即可。因子、回测应使用相应模块提供的请求类型和上下文。

| 常用上下文成员 | 使用场景 |
|---|---|
| `Mode` | 选择 `"PRD"` 或 `"DEBUG"`，具体调试行为由业务节点实现 |
| `DataCache` | 提供已启动、与节点兼容的缓存对象 |
| `DataVersion` | 声明数据快照版本，供支持缓存的节点隔离或复用结果 |
| `Resources` | 注入资源服务或句柄，供自定义节点读取 |
| `EventHandler` | 接收执行进度事件的回调函数 |
| `NodeErrors` | 在引擎 `FailMode="skip"` 时查询失败原因 |

每次 `run()` 都开始新的一轮运行，清理上轮节点临时状态，保留显式资源和数据缓存。需要跨运行延续的业务状态，应交给专门的业务对象或持久化机制。

排查问题时，`QSID` 表示计算定义，`InstanceID` 区分节点实例，`context.RunID` 区分运行。详细区别见[原理和细节](原理和细节.ipynb)。

## 自定义节点需要哪些方法

引擎按“初始化 → 准备 → 计算”组织任务。简单计算沿用默认流程即可；有数据加载、请求转换或分片合并需要时，再扩展对应方法。

| 方法 | 何时扩展 |
|---|---|
| `backward_compute()` | 实现实际计算，通常必须重写 |
| `init_compute()` | 汇总请求范围、登记临时状态或准备任务 |
| `prepare_compute()` | 加载数据等准备工作；只执行已登记的任务 |
| `forward_compute()` | 为依赖生成不同请求，或传递局部上下文 |
| `merge_result()` | 合并数据分片并行产生的多个结果 |

`forward_compute()` 返回的第二项会作为 `local_context` 传给当前节点的 `backward_compute()`。可以使用普通对象，也可以使用 `LocalContext`；时序节点另有 `DTInitData`、`DTLocalContext`。完整签名、准备任务注册和构造钩子见[原理和细节](原理和细节.ipynb)。

建议把扩展逻辑放在这些生命周期方法中。`StackEngine`、`TreeEngine` 直接编排前向和后向方法，仅重写 `compute()` 的逻辑不会在所有引擎下生效。

## 查看计算图

`node2dict()` 和 `dict2mermaid()` 生成 Mermaid 图文本，`mermaid-python` 可用于显示计算图。`direction="TD"` 表示从上到下，也可用 `"LR"`、`"BT"`、`"RL"`。

这张图按计算语义合并节点，适合检查依赖结构；它不区分计算定义相同的多个实例，也不展示实时执行状态。相关边界见[原理和细节](原理和细节.ipynb)。

In [6]:
from QuantStudio.Tools.Visualization import node2dict, dict2mermaid

graph_text = dict2mermaid(node2dict([configured]), direction="TD")

In [7]:
from IPython.display import display
from mermaid import Mermaid

display(Mermaid(graph_text))